# Prepare SPARCS Data Split

This notebook prepares the 2024 SPARCS inpatient discharge dataset for prolonged length of stay prediction.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
import json
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split

In [ ]:
BASE_DIR = "/content/drive/MyDrive/FYP/SPARCS"

DATA_PATH = "/content/drive/MyDrive/FYP/SPARCS/data/SPARCS_dataset.csv"

PROCESSED_DIR = f"{BASE_DIR}/processed"
SPLIT_DIR = f"{BASE_DIR}/splits"
RESULT_DIR = f"{BASE_DIR}/results"

os.makedirs(PROCESSED_DIR, exist_ok=True)
os.makedirs(SPLIT_DIR, exist_ok=True)
os.makedirs(RESULT_DIR, exist_ok=True)

In [ ]:
df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
df.head()

/tmp/ipykernel_5567/774394781.py:1: DtypeWarning: Columns (29) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(DATA_PATH)


Dataset shape: (2196737, 33)


,Health Service Area,Hospital County,Operating Certificate Number,Permanent Facility Id,Facility Name,Age Group,Zip Code,Gender,Race,Ethnicity,Length of Stay,Type of Admission,Patient Disposition,Discharge Year,CCSR Diagnosis Code,CCSR Diagnosis Description,CCSR Procedure Code,CCSR Procedure Description,APR DRG Code,APR DRG Description,APR MDC Code,APR MDC Description,APR Severity of Illness Code,APR Severity of Illness Description,APR Risk of Mortality,APR Medical Surgical Description,Payment Typology 1,Payment Typology 2,Payment Typology 3,Birth Weight,Emergency Department Indicator,Total Charges,Total Costs
0,Hudson Valley,Westchester,5957001.0,1139.0,WESTCHESTER MEDICAL CENTER,0-17,OOS,F,White,Not Span/Hispanic,1,Emergency,Home or Self Care,2024,SYM002,FEVER,NaN,NaN,722,FEVER AND INFLAMMATORY CONDITIONS,18,"INFECTIOUS AND PARASITIC DISEASES, SYSTEMIC OR...",2,Moderate,Minor,Medical,Private Health Insurance,NaN,NaN,NaN,Y,46814.00,6772.07
1,New York City,Queens,7003001.0,1628.0,FLUSHING HOSPITAL MEDICAL CENTER,0-17,113,M,White,Spanish/Hispanic,2,Emergency,Home or Self Care,2024,SYM002,FEVER,NaN,NaN,722,FEVER AND INFLAMMATORY CONDITIONS,18,"INFECTIOUS AND PARASITIC DISEASES, SYSTEMIC OR...",2,Moderate,Moderate,Medical,Medicaid,NaN,NaN,NaN,Y,13490.00,15464.30
2,New York City,New York,7002054.0,1458.0,NEW YORK-PRESBYTERIAN HOSPITAL - NEW YORK WEIL...,70 or Older,100,M,White,Not Span/Hispanic,2,Emergency,Home or Self Care,2024,SYM002,FEVER,ADM012,CHEMOTHERAPY,722,FEVER AND INFLAMMATORY CONDITIONS,18,"INFECTIOUS AND PARASITIC DISEASES, SYSTEMIC OR...",2,Moderate,Moderate,Medical,Medicare,Private Health Insurance,NaN,NaN,Y,49503.16,9324.77
3,New York City,New York,7002054.0,1464.0,NEW YORK-PRESBYTERIAN HOSPITAL - COLUMBIA PRES...,0-17,100,F,Other Race,Not Span/Hispanic,1,Emergency,Home or Self Care,2024,SYM002,FEVER,CNS002,LUMBAR PUNCTURE,722,FEVER AND INFLAMMATORY CONDITIONS,18,"INFECTIOUS AND PARASITIC DISEASES, SYSTEMIC OR...",1,Minor,Minor,Medical,Private Health Insurance,NaN,NaN,2700,Y,27827.66,7304.27
4,New York City,New York,7002032.0,1466.0,MOUNT SINAI WEST,18-29,100,F,Other Race,Spanish/Hispanic,1,Emergency,Home or Self Care,2024,SYM002,FEVER,ADM021,"ADMINISTRATION OF THERAPEUTIC SUBSTANCES, NEC",722,FEVER AND INFLAMMATORY CONDITIONS,18,"INFECTIOUS AND PARASITIC DISEASES, SYSTEMIC OR...",2,Moderate,Minor,Medical,Medicare,NaN,NaN,NaN,Y,32798.29,7948.10


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2196737 entries, 0 to 2196736
Data columns (total 33 columns):
 #   Column                               Dtype  
---  ------                               -----  
 0   Health Service Area                  object 
 1   Hospital County                      object 
 2   Operating Certificate Number         float64
 3   Permanent Facility Id                float64
 4   Facility Name                        object 
 5   Age Group                            object 
 6   Zip Code                             object 
 7   Gender                               object 
 8   Race                                 object 
 9   Ethnicity                            object 
 10  Length of Stay                       object 
 11  Type of Admission                    object 
 12  Patient Disposition                  object 
 13  Discharge Year                       int64  
 14  CCSR Diagnosis Code                  object 
 15  CCSR Diagnosis Description      

In [ ]:
df.columns.tolist()

['Health Service Area',
 'Hospital County',
 'Operating Certificate Number',
 'Permanent Facility Id',
 'Facility Name',
 'Age Group',
 'Zip Code',
 'Gender',
 'Race',
 'Ethnicity',
 'Length of Stay',
 'Type of Admission',
 'Patient Disposition',
 'Discharge Year',
 'CCSR Diagnosis Code',
 'CCSR Diagnosis Description',
 'CCSR Procedure Code',
 'CCSR Procedure Description',
 'APR DRG Code',
 'APR DRG Description',
 'APR MDC Code',
 'APR MDC Description',
 'APR Severity of Illness Code',
 'APR Severity of Illness Description',
 'APR Risk of Mortality',
 'APR Medical Surgical Description',
 'Payment Typology 1',
 'Payment Typology 2',
 'Payment Typology 3',
 'Birth Weight',
 'Emergency Department Indicator',
 'Total Charges',
 'Total Costs']

In [ ]:
# Clean Length of Stay
# SPARCS LOS may be stored as text.
# This converts it to numeric days.

df["los_numeric"] = (
    df["Length of Stay"]
    .astype(str)
    .str.replace("+", "", regex=False)
    .str.strip()
)

df["los_numeric"] = pd.to_numeric(df["los_numeric"], errors="coerce")

print(df["los_numeric"].isna().sum())
df[["Length of Stay", "los_numeric"]].head()

0


,Length of Stay,los_numeric
0,1,1
1,2,2
2,2,2
3,1,1
4,1,1


In [ ]:
df = df.dropna(subset=["los_numeric"])

In [ ]:
# Define Target
# Prolonged LOS = Length of Stay >= 7 days

df["prolonged_los"] = (df["los_numeric"] >= 7).astype(int)
df["prolonged_los"].value_counts()

,count
prolonged_los,
0,1667025
1,529712


In [ ]:
positive_rate = df["prolonged_los"].mean()
print(f"Positive class rate: {positive_rate:.4f}")

Positive class rate: 0.2411


In [ ]:
# Define Deployment-Friendly Feature Schema

FEATURE_SCHEMA = {
    "Age Group": {
        "label": "Age Group",
        "description": "Patient age category at the time of discharge.",
        "input_type": "select"
    },
    "Gender": {
        "label": "Gender",
        "description": "Patient gender recorded in the discharge record.",
        "input_type": "select"
    },
    "Race": {
        "label": "Race",
        "description": "Patient race category recorded in the dataset.",
        "input_type": "select"
    },
    "Ethnicity": {
        "label": "Ethnicity",
        "description": "Patient ethnicity category recorded in the dataset.",
        "input_type": "select"
    },
    "Type of Admission": {
        "label": "Admission Type",
        "description": "Indicates whether the admission was emergency, urgent, elective, or another type.",
        "input_type": "select"
    },
    "CCSR Diagnosis Description": {
        "label": "Diagnosis Category",
        "description": "Clinical diagnosis category assigned to the inpatient discharge.",
        "input_type": "select"
    },
    "CCSR Procedure Description": {
        "label": "Procedure Category",
        "description": "Procedure category associated with the inpatient discharge.",
        "input_type": "select"
    },
    "APR DRG Description": {
        "label": "APR DRG",
        "description": "Diagnosis-related group describing the inpatient case type.",
        "input_type": "select"
    },
    "APR MDC Description": {
        "label": "Major Diagnosis Category",
        "description": "Broad clinical category related to the inpatient stay.",
        "input_type": "select"
    },
    "APR Severity of Illness Description": {
        "label": "Severity of Illness",
        "description": "Severity level assigned to the inpatient case.",
        "input_type": "select"
    },
    "APR Risk of Mortality": {
        "label": "Risk of Mortality",
        "description": "Mortality risk level assigned to the inpatient case.",
        "input_type": "select"
    },
    "APR Medical Surgical Description": {
        "label": "Medical or Surgical Case",
        "description": "Indicates whether the inpatient stay was medical or surgical.",
        "input_type": "select"
    },
    "Emergency Department Indicator": {
        "label": "Emergency Department Visit",
        "description": "Indicates whether the patient came through the emergency department.",
        "input_type": "select"
    }
}

FEATURE_COLUMNS = list(FEATURE_SCHEMA.keys())

In [ ]:
# Generate Dropdown Options From Dataset
# Since some features have many categories, therefore set it choose from the dataset

for feature in FEATURE_SCHEMA:
  options = (
      df[feature]
      .dropna()
      .astype(str)
      .sort_values()
      .unique()
      .tolist()
  )

  FEATURE_SCHEMA[feature]["options"] = [
      {"label": value, "value": value}
      for value in options
  ]


In [ ]:
# Create Feature Schema Table

feature_schema_df = pd.DataFrame([
    {
        "feature": feature,
        "label": details["label"],
        "description": details["description"],
        "input_type": details["input_type"],
        "options": json.dumps(details.get("options", []))
    }
    for feature, details in FEATURE_SCHEMA.items()
])

feature_schema_df

,feature,label,description,input_type,options
0,Age Group,Age Group,Patient age category at the time of discharge.,select,"[{""label"": ""0-17"", ""value"": ""0-17""}, {""label"":..."
1,Gender,Gender,Patient gender recorded in the discharge record.,select,"[{""label"": ""F"", ""value"": ""F""}, {""label"": ""M"", ..."
2,Race,Race,Patient race category recorded in the dataset.,select,"[{""label"": ""Black/African American"", ""value"": ..."
3,Ethnicity,Ethnicity,Patient ethnicity category recorded in the dat...,select,"[{""label"": ""Multi-ethnic"", ""value"": ""Multi-eth..."
4,Type of Admission,Admission Type,"Indicates whether the admission was emergency,...",select,"[{""label"": ""Elective"", ""value"": ""Elective""}, {..."
5,CCSR Diagnosis Description,Diagnosis Category,Clinical diagnosis category assigned to the in...,select,"[{""label"": ""ABDOMINAL HERNIA"", ""value"": ""ABDOM..."
6,CCSR Procedure Description,Procedure Category,Procedure category associated with the inpatie...,select,"[{""label"": ""ABDOMINAL WALL PROCEDURES, NEC"", ""..."
7,APR DRG Description,APR DRG,Diagnosis-related group describing the inpatie...,select,"[{""label"": ""ABDOMINAL PAIN"", ""value"": ""ABDOMIN..."
8,APR MDC Description,Major Diagnosis Category,Broad clinical category related to the inpatie...,select,"[{""label"": ""ALCOHOL/DRUG USE AND ALCOHOL/DRUG ..."
9,APR Severity of Illness Description,Severity of Illness,Severity level assigned to the inpatient case.,select,"[{""label"": ""Extreme"", ""value"": ""Extreme""}, {""l..."


In [ ]:
# Prepare X and y

X = df[FEATURE_COLUMNS].copy()
y = df["prolonged_los"].copy()

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (2196737, 13)
y shape: (2196737,)


In [ ]:
# Train / Validation / Test Split
# Since SPARCS public data has no patient identifier, use stratified split

all_idx = np.arange(len(df))

train_val_idx, test_idx = train_test_split(
    all_idx,
    test_size=0.20,
    random_state=42,
    stratify=y
)

y_train_val = y.iloc[train_val_idx]

train_idx, val_idx = train_test_split(
    train_val_idx,
    test_size=0.20,
    random_state=42,
    stratify=y_train_val
)

In [ ]:
def split_summary(name, indices):
    subset_y = y.iloc[indices]

    return {
        "split": name,
        "rows": len(indices),
        "positive_rate": subset_y.mean(),
        "positive_count": int(subset_y.sum()),
        "negative_count": int((subset_y == 0).sum())
    }

summary_df = pd.DataFrame([
    split_summary("train", train_idx),
    split_summary("validation", val_idx),
    split_summary("test", test_idx)
])

summary_df

,split,rows,positive_rate,positive_count,negative_count
0,train,1405911,0.241135,339015,1066896
1,validation,351478,0.241136,84754,266724
2,test,439348,0.241137,105943,333405


In [ ]:
# Save processed Dataset, Splits, Schema

processed_path = f"{PROCESSED_DIR}/sparcs_2024_processed.csv"

df.to_csv(processed_path, index=False)

np.save(f"{SPLIT_DIR}/train_idx.npy", train_idx)
np.save(f"{SPLIT_DIR}/val_idx.npy", val_idx)
np.save(f"{SPLIT_DIR}/test_idx.npy", test_idx)

joblib.dump(FEATURE_COLUMNS, f"{SPLIT_DIR}/feature_columns.joblib")
joblib.dump(FEATURE_SCHEMA, f"{SPLIT_DIR}/feature_schema.joblib")

summary_df.to_csv(f"{RESULT_DIR}/split_summary.csv", index=False)
feature_schema_df.to_csv(f"{RESULT_DIR}/feature_schema.csv", index=False)

print("Saved processed dataset, split files, and feature schema.")
print("Processed file:", processed_path)

Saved processed dataset, split files, and feature schema.
Processed file: /content/drive/MyDrive/FYP/SPARCS/processed/sparcs_2024_processed.csv
